In [ ]:
from os.path import join
import sys
from glue.core.link_helpers import LinkSame

import numpy as np

from glue.core import Data
from glue.core.subset import AndState, CategorySubsetState, RangeSubsetState

from glue_jupyter import jglue
from glue_jupyter.bqplot.histogram import BqplotHistogramView
from glue_jupyter.bqplot.scatter import BqplotScatterView

In [ ]:
from os.path import splitext
import string

from glue.core.link_helpers import LinkSame


def printable_string(s):
    return "".join(filter(lambda x: x in string.printable, s))


# Some of the components have non-printable characters in their names
# when loaded from the CSV, which makes them hard to grab later.
# So we strip any non-printable characters from the component names
def clean_component_names(data):
    for component in data.components:
        component.label = printable_string(component.label)


def layer_for_data(viewer, data):
    for layer in viewer.layers:
        if layer.layer is data:
            return layer
    return None


# TODO: This is not very robust, just to get things working
def color_for_path(filepath, colors):
    name, ext = splitext(filepath)
    if ext != ".csv":
        return None 
    for color in colors:
        color = str(color)
        if color.lower() in name.lower():
            return color

    return None

def color_filename(color):
    return f"GenEd 1112 {color.upper()}.csv"

In [ ]:
app = jglue()
dc = app.data_collection

In [ ]:
data_filepath_2025 = "2025_student_results.csv"
data = app.load_data(data_filepath_2025)
clean_component_names(data)
data.style.color = "#ffffff"
colors = np.unique(data["Color"])
for color, code in zip(colors, colors.codes) :
    dc.new_subset_group(
        label=color,
        subset_state=CategorySubsetState(att=data.id["Color"], categories=[code]),
        color=color,
        alpha=1
    )

In [ ]:
# Make the "1-1" data
# and link it to our main data set
one_to_one = Data(label="one-to-one", Uncertainty=[0, 1000], End=[0, 1000])
dc.append(one_to_one)
dc.add_link(LinkSame(data.id["UNCERTAINTY"], one_to_one.id["Uncertainty"]))
dc.add_link(LinkSame(data.id["End_off_feet"], one_to_one.id["End"]))

In [ ]:
# Find the winner(s) and create a subset
min_distance = min(data["End_off_feet"])
winner_sg = dc.new_subset_group(
    label="Winners 2025",
    subset_state=RangeSubsetState(lo=min_distance, hi=min_distance,
                                 att=data.id["End_off_feet"]),
    color="Purple"
)

In [ ]:
# Make a subset for the correct end
lat = data["End_Lat_true"][0]
lon = data["End_Long_true"][0]
lat_ss = RangeSubsetState(lo=lat, hi=lat, att=data.id["EndLat"])
lon_ss = RangeSubsetState(lo=lon, hi=lon, att=data.id["EndLong"])
endpoint_sg = dc.new_subset_group(label="Common End",
                                    subset_state=AndState(lat_ss, lon_ss),
                                    color="Red")

In [ ]:
locations_scatter = app.new_data_viewer(BqplotScatterView, data=data)
locations_scatter.state.x_att = data.id["EndLong"]
locations_scatter.state.y_att = data.id["EndLat"]
for layer in locations_scatter.layers:
    if layer.layer.label == "Common End":
        layer.state.size = 12
        layer.state.alpha = 1

In [ ]:
stats_histogram = app.new_data_viewer(BqplotHistogramView, data=data)
stats_histogram.state.x_att = data.id["Color"]
stats_histogram.hist_n_bins = 5